### 1. Instalação de Dependências (Se necessário)

**Preparação do Ambiente (Serverless)**

> Instalação das bibliotecas necessárias para a execução do pipeline no ambiente Databricks Serverless. 
* **adlfs:** Necessário para permitir que o Pandas converse diretamente com o protocolo `abfs://` do Azure Data Lake.
* **python-dotenv:** Garante a leitura segura de credenciais locais ou injetadas via CI/CD.

In [0]:
%pip install adlfs pandas python-dotenv

### 2. Imports e Carga do .env

**Gestão de Segurança e Variáveis de Ambiente**

> Nesta etapa, importou-se os módulos necessários e carregamos o arquivo `.env`.

**Decisão Arquitetural:** Ao em vez de hardcodar senhas no notebook (o que fere as políticas de segurança), utilizamos o `python-dotenv` com uma busca recursiva para localizar as credenciais independentemente do diretório de execução do cluster.

In [0]:
import os
import pandas as pd
import adlfs
from dotenv import load_dotenv

# 1. LOCALIZAÇÃO E CARGA DO ARQUIVO .ENV
caminho_env = None
for tentativa in [".env", "../.env", "../../.env"]:
    if os.path.exists(tentativa):
        caminho_env = tentativa
        break

if caminho_env:
    load_dotenv(dotenv_path=caminho_env)
    print(f"✅ Arquivo .env localizado e carregado com sucesso de: '{caminho_env}'")
else:
    raise FileNotFoundError("⚠️ ERRO CRÍTICO: O arquivo .env não foi localizado.")

### 3. Mapeamento de Segurança e Data Discovery (Varredura)

**Data Discovery e Mapeamento Dinâmico no ADLS**

> Como os arquivos de origem podem sofrer alterações de extensão (ex: `.csv` para `.parquet`) ou possuir variações de nomenclatura, implementou-se uma varredura dinâmica no container `real-time-ecommerce-data`.

**Lógica Implementada:**

1. Autenticação via Azure SDK (Client Credentials).
2. Varredura com filtro exato para `ecommerce_pedidos` (evitando colisão com tabelas de itens).
3. Identificação e armazenamento dinâmico da extensão do arquivo para a etapa de extração.

In [0]:
# 2. RESGATE DE CREDENCIAIS E MAPEAMENTO
tenant_id = os.getenv("ADLS_TENANT_ID")
client_id = os.getenv("ADLS_CLIENT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account = "internshipdatalake"
container_name = "real-time-ecommerce-data"

if not tenant_id or not client_id or not client_secret:
    raise ValueError("⚠️ ERRO: As credenciais do Data Lake estão vazias no arquivo .env.")

# Injeção das variáveis exigidas pelo Azure SDK
os.environ["AZURE_TENANT_ID"] = tenant_id
os.environ["AZURE_CLIENT_ID"] = client_id
os.environ["AZURE_CLIENT_SECRET"] = client_secret

# 3. DESCOBERTA AUTOMÁTICA COM IDENTIFICAÇÃO DE FORMATO
fs = adlfs.AzureBlobFileSystem(
    account_name=storage_account,
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)

print("🔍 Iniciando varredura inteligente no Data Lake...")
todos_arquivos = fs.find(container_name)

caminho_real_no_azure = None
formato_detectado = None

# BOA PRÁTICA: Buscamos pelo termo exato 'ecommerce_pedidos', evitando o conflito com 'ecommerce_itens_pedido'
for arquivo in todos_arquivos:
    if "ecommerce_pedidos" in arquivo.lower() and "itens_pedido" not in arquivo.lower():
        caminho_real_no_azure = arquivo
        
        # Identifica a extensão do arquivo dinamicamente (pega tudo após o último ponto)
        formato_detectado = arquivo.split(".")[-1].lower()
        break

if not caminho_real_no_azure:
    raise FileNotFoundError("⚠️ ERRO CRÍTICO: Nenhum arquivo correspondente à tabela 'ecommerce_pedidos' foi localizado.")

caminho_csv_final = f"abfs://{caminho_real_no_azure}"
print(f"📌 Arquivo localizado com sucesso: {caminho_csv_final}")
print(f"📦 Formato identificado automaticamente: {formato_detectado.upper()}")

### 4. Extração Dinâmica e Conversão para PySpark

> Devido às restrições de injeção de configurações globais (`spark.conf`) no ambiente Serverless via Spark Connect, adotou-se uma estratégia de bypass seguro:

* **Leitura:** O Pandas utiliza as credenciais encapsuladas (`storage_options`) para extrair os dados do ADLS para a memória da máquina de forma agnóstica ao formato.
* **Conversão:** O DataFrame local é convertido em um PySpark DataFrame para habilitar o processamento distribuído nas próximas camadas da arquitetura.

In [0]:
# ==========================================
# TRAVA DE SEGURANÇA PARA VALIDAÇÃO (CODE REVIEW)
# ==========================================
if 'formato_detectado' not in locals() and 'formato_detectado' not in globals():
    raise NameError(
        "⚠️ ERRO DE SEQUÊNCIA NA VALIDAÇÃO: A variável 'formato_detectado' não foi encontrada. "
        "Por favor, execute a Célula 3 (Varredura do Data Lake) imediatamente antes de rodar esta célula!"
    )

# ==========================================
# 4. EXTRAÇÃO DINÂMICA (PANDAS BYPASS)
# ==========================================
print(f"📥 Iniciando extração dinâmica do formato: {formato_detectado.upper()}...")

# CRIANDO O CRACHÁ DE ACESSO PARA O PANDAS
# Recuperamos as variáveis da Célula 3 e empacotamos para o Pandas usar nos bastidores
credenciais_pandas = {
    "account_name": storage_account,
    "client_id": client_id,
    "client_secret": client_secret,
    "tenant_id": tenant_id
}

# O Pandas agora recebe o 'storage_options' para provar que tem acesso ao arquivo
if formato_detectado == "parquet":
    df_pandas = pd.read_parquet(caminho_csv_final, storage_options=credenciais_pandas)
    
elif formato_detectado == "csv":
    df_pandas = pd.read_csv(caminho_csv_final, storage_options=credenciais_pandas)
    
elif formato_detectado == "json":
    df_pandas = pd.read_json(caminho_csv_final, storage_options=credenciais_pandas)
    
else:
    raise TypeError(f"⚠️ FORMATO NÃO SUPORTADO: O formato '.{formato_detectado}' não possui um leitor configurado.")

# ==========================================
# 5. CONVERSÃO PARA COMPUTAÇÃO DISTRIBUÍDA
# ==========================================
print("📊 Convertendo a estrutura para PySpark DataFrame...")
df_pedidos = spark.createDataFrame(df_pandas)

print("✅ Dados extraídos e prontos no Spark! Veja a estrutura do schema:")

### 5. Carga Final (Data Load) no Azure SQL Server
> Gravação dos dados estruturados no banco de dados relacional oficial do projeto.

**Detalhes da Conexão:**
* Utiliza o conector nativo homologado pela Databricks (`.format("sqlserver")`), eliminando incompatibilidades da string JDBC genérica no ambiente Serverless.
* Os dados são gravados com modo `overwrite` na tabela oficial da equipe: `squad2.ecommerce_pedidos`.

In [0]:
# ==========================================
# 6. CONFIGURAÇÃO DO BANCO DE DADOS
# ==========================================
import os

jdbc_host = os.getenv("SQL_HOST")
jdbc_db = os.getenv("SQL_DATABASE")
jdbc_user = os.getenv("SQL_USERNAME")
jdbc_pass = os.getenv("SQL_PASSWORD")

# O schema 'squad2' agora existe oficialmente no banco de dados!
nome_tabela_destino = "squad2.ecommerce_pedidos"

print(f"📤 Gravando os dados na tabela destino: '{nome_tabela_destino}' no SQL Server...")

# ==========================================
# 7. CARGA VIA CONECTOR NATIVO (FORMATAÇÃO RECUADA)
# ==========================================
(
    df_pedidos.write
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_db)
    .option("user", jdbc_user)
    .option("password", jdbc_pass)
    .option("dbtable", nome_tabela_destino)
    .option("encrypt", "true")
    .option("trustServerCertificate", "false")
    .mode("overwrite")
    .save()
)

print("🚀 SUCESSO ABSOLUTO! O pipeline finalizou a carga no schema correto do SQL Server!")